In [ ]:
import pandas as pd
df = pd.read_csv("registro_experimento.csv")
df.head()
df.groupby("ubicacion")["ventas_producto"].mean()
df["ubicacion"].value_counts()

,count
ubicacion,
pasillo_principal,2
habitual,2


In [ ]:
import pandas as pd
import numpy as np
import json
from scipy import stats

tiendas = ["Quit Norte", "Quito sur", "Guayaquil centro", "Guayaquil norte", "Cuenca centro", "Cuenca sur", "ambato centro", "Manta", "Loja", "Machala"]
semanas = [1, 2, 3, 4]

rng = np.random.default_rng(1234597)
primera = rng.choice(["pasillo_principal", "habitual"], size=len(tiendas))

filas = []
# 1. Primero construimos todas las filas de todas las tiendas y semanas
for tienda, cond in zip(tiendas, primera):
    otra = "habitual" if cond == "pasillo_principal" else "pasillo_principal"
    for s in semanas:
        filas.append({
            "tienda": tienda,
            "semana": s,
            "ubicacion": cond if s % 2 == 1 else otra
        })

# 2. Fuera de los bucles, creamos el DataFrame completo
diseno = pd.DataFrame(filas)
print(diseno.head())

# 3. Simulamos las ventas para todo el DataFrame
rng = np.random.default_rng(1234598)
efecto_tienda = dict(zip(tiendas, rng.normal(1100, 120, len(tiendas))))
#print(efecto_tienda)
efecto_semana = {1: 0, 2: 30, 3: 60, 4: 90}

diseno["ventas_producto"] = (
    diseno["tienda"].map(efecto_tienda) +
    diseno["semana"].map(efecto_semana) +
    (diseno["ubicacion"] == "pasillo_principal") * 220 +
    rng.normal(0, 90, len(diseno))
).round(0).astype(int)

# 4. Promoción simultánea registrada
# 4. Promoción simultánea registrada
diseno["promocion_simulacion"] = "no"
diseno.loc[(diseno["tienda"] == "Guayaquil centro") & (diseno["semana"] == 1), "promocion_simulacion"] = "si"
diseno.loc[diseno["promocion_simulacion"] == "si", "ventas_producto"] += 180

diseno.to_csv("resultado_simulacion.csv", index=False)

# 5. Análisis pareado: una fila por tienda con sus condiciones
piv = diseno.groupby(["tienda", "ubicacion"])["ventas_producto"].mean().unstack()
print("Columnas de la tabla pivote:", piv.columns)

# Corregimos la resta con el operador '-' y aseguramos las dos columnas
piv["diferencia"] = piv["pasillo_principal"] - piv["habitual"]

media = piv["diferencia"].mean()
dev = piv["diferencia"].std(ddof=1)

print("Media de las diferencias:", media)
print("Desviación estándar:", dev)
n =  len(piv)

t_critic = stats.t.ppf(0.975, df=9)
ic_inf = media - (t_critic * dev / n**0.5)
ic_sup = media + (t_crict*dev/n**0.5)
print(ic_inf)

print(f"[{ic_inf} , {ic_sup}]")

prueba = stats.ttest_rel(piv["pasillo_principal"], piv["habitual"])
print(f"Prueba t: {prueba.pvalue: .10f}")

       tienda  semana          ubicacion
0  Quit Norte       1           habitual
1  Quit Norte       2  pasillo_principal
2  Quit Norte       3           habitual
3  Quit Norte       4  pasillo_principal
4   Quito sur       1  pasillo_principal
Columnas de la tabla pivote: Index(['habitual', 'pasillo_principal'], dtype='object', name='ubicacion')
Media de las diferencias: 239.7
Desviación estándar: 102.39471557545232
166.45123307638832
[166.45123307638832 , 312.94876692361163]
Prueba t:  0.0000409166


In [ ]:
%load_ext rpy2.ipython

In [ ]:
%%R
df <- read.csv("resultado_simulacion.csv", stringsAsFactors = FALSE)
print(aggregate(ventas_producto ~ ubicacion, data = df, FUN=mean))
print(df[, c("tienda", "semana", "ubicacion", "ventas_producto", "promocion_simulacion")])
media_tienda <- tapply(df$ventas_producto, list(df$tienda, df$ubicacion), mean)
print(round(media_tienda, 0))

diferencias <- media_tienda[, "pasillo_principal"] - media_tienda[, "habitual"]
print(diferencias)

prueba <- t.test(media_tienda[,"pasillo_principal"], media_tienda[,"habitual"],paired = TRUE)
print(prueba$conf.int)
print(prueba$p.value)

          ubicacion ventas_producto
1          habitual          1107.2
2 pasillo_principal          1346.9
             tienda semana         ubicacion ventas_producto
1        Quit Norte      1          habitual             979
2        Quit Norte      2 pasillo_principal            1380
3        Quit Norte      3          habitual            1055
4        Quit Norte      4 pasillo_principal            1259
5         Quito sur      1 pasillo_principal            1158
6         Quito sur      2          habitual             978
7         Quito sur      3 pasillo_principal            1359
8         Quito sur      4          habitual            1058
9  Guayaquil centro      1          habitual            1407
10 Guayaquil centro      2 pasillo_principal            1569
11 Guayaquil centro      3          habitual            1326
12 Guayaquil centro      4 pasillo_principal            1668
13  Guayaquil norte      1 pasillo_principal            1331
14  Guayaquil norte      2          ha